# Check Exposure On BDB2026 Data For WRs

BDB2027 data does not allow us to examine how offensive players and defensive players interact. 

However, we could use data from BDB2026 to see how receivers and defenders interact during ***pass plays from the 2023 REG Season***. 

The advantage of using 2026 BDB data is that we would be able to continuously see how far away receivers are from defenders (and vice versa) while routes are being run. This would allow us to see how much separation receivers create and if metrics that we create from the BDB 2027 Combine data are associated with separation. Similarly, we would be able to see if Combine metrics from cornerbacks are associated with them being able to stay with the receiver more. 

The disadvantage is that our 2027 BDB data only gives us data on players who entered the league from 2023-2025 while the 2026 BDB data gives us data on routes from 2023 alone. Therefore, the only players we would be able to examine are ***rookie receivers and cornerbacks from the 2023 season***.

Therefore, in this file we check if there would be enough data on 2023 rookie receivers and cornerbacks from our 2026 and 2027 BDB data to have a worthwhile project.

<br><br/>
#### *Initialize Environment*

In [ ]:
import sys
from pathlib import Path

# Identify Project Root
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

# Import Shared Project Paths
from utils.paths import (
    CLEAN_DATA_DIR,
    BDB_2026_BEFORE_PASS_DIR,
    BDB_2026_SUPPLEMENTARY_DIR,
)

import polars as pl 
import pandas as pd 
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"

# Import schema_map For 2027 BDB Data
import sys
from pathlib import Path
# Identify Project Root

PROJECT_ROOT = next(

    parent

    for parent in [Path.cwd(), *Path.cwd().parents]

    if (parent / "pyproject.toml").exists()

)



# Import Shared Data Schemas

sys.path.insert(0, str(PROJECT_ROOT))

from utils.schema import schema_map

<br><br/>
## Load Data
***

#### Players Data (*BDB 2027*)

In [ ]:
# =============================
# Players Data
# =============================

players_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "players.csv"), 
        schema_overrides = schema_map["players"]
    )
)

# -----------------------------
# Only Keep Position of Interest
# -----------------------------

position_of_interest = "WR"

players_df = (
    players_df
    .filter(pl.col("nfl_position") == position_of_interest)
)

# -----------------------------
# Only Keep 2023 Rookies 
# -----------------------------

# We're only keeping 2023 rookies because the 2026 BDB
# player-tracking data only include 2023 REG season

players_df = (
    players_df
    .filter(
        pl.col("draft_year") == 2023
    )
)

# -----------------------------
# Only Keep Needed Variables
# -----------------------------
players_df = (
    players_df
    .select(
        "nfl_id", 
        "display_name", 
        "draft_year", 
        "nfl_position"
    )
)

# -----------------------------
# Identify Players of Interest
# -----------------------------

players_of_interest = (
    players_df
    .select("nfl_id")
    .unique()
    .to_series()
    .to_list()    
)

print("Players Data:")
print(f"    - Number of Players From 2023 Rookie Class: {len(players_of_interest)}")
players_df

<br><br/>
#### Combine Results Data (*BDB 2027*)

In [ ]:
# =============================
# Combine Results Data
# =============================

combine_results_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "combine_results.csv"), 
        schema_overrides = schema_map["combine_results"]
    )
)

# -----------------------------
# Only Keep Players of Interest
# -----------------------------

combine_results_df = (
    combine_results_df
    .filter(pl.col("nfl_id").is_in(players_of_interest))
)

# -----------------------------
# Drop Unneeded Columns
# -----------------------------

combine_results_df = (
    combine_results_df
    .drop("draft_year")
)

print("Combine Results Data:")
print(f"    - Number of Players In This Dataset: {combine_results_df.select(pl.col("nfl_id")).n_unique()}")
combine_results_df

<br><br/>
#### Combine Tracking Data (*BDB 2027*)

In [ ]:
# =============================
# Combine Tracking Data
# =============================

combine_tracking_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "combine_tracking.csv"), 
        schema_overrides = schema_map["combine_tracking"]
    )
)

# -----------------------------
# Only Keep Players of Interest
# -----------------------------

combine_tracking_df = (
    combine_tracking_df
    .filter(pl.col("nfl_id").is_in(players_of_interest))
)

# -----------------------------
# Don't Keep Ball Tracking Data
# -----------------------------

combine_tracking_df = (
    combine_tracking_df
    .filter(pl.col("entity_type") == "PLAYER")
)

print("Combine Tracking Data:")
print(f"    - Number of Players In This Dataset: {combine_tracking_df.select(pl.col("nfl_id")).n_unique()}")
combine_tracking_df

<br><br/>
#### Player-Play Data (*BDB 2027*)

In [ ]:
# =============================
# Player-Play Data
# =============================

player_play_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "player_play.csv"), 
        schema_overrides = schema_map["combine_tracking"]
    )
)

# -----------------------------
# Only Keep Players of Interest
# -----------------------------

player_play_df = (
    player_play_df
    .filter(pl.col("nfl_id").is_in(players_of_interest))
)

# ----------------------------
# Only Keep Helpful Variables
# ----------------------------

player_play_df = (
    player_play_df
    .select(
        "game_id", 
        "play_id", 
        "nfl_id", 
        "target"
    )
)

print("Player-Play Data:")
print(f"    - Number of Players In This Dataset: {player_play_df.select(pl.col("nfl_id")).n_unique()}")
player_play_df

<br><br/>
#### Supplemental Data (*BDB 2026*)

In [ ]:
# ==================================
# Load Supplemental Data
# ==================================

supp_data = (
    pl.read_csv(
        str(BDB_2026_SUPPLEMENTARY_DIR / "supplementary_data.csv"), 
        # Change "NA" to None
        null_values = {
            "play_action": "NA", 
            "dropback_distance": "NA"
        }
    )
)

# ---------------------------------
# Only Keep Helpful Variables
# ---------------------------------

supp_data = (
    supp_data
    .select(
        # ID Variables
        "game_id", 
        "play_id", 
        # Helpful Variables
        "play_description", 
        "route_of_targeted_receiver", 
        "team_coverage_man_zone", 
        "team_coverage_type", 
        "pass_length", 
        "yardline_number", 
        "pass_result", 
        "play_nullified_by_penalty"
    )
)

print("Supplemental Data:")
supp_data

<br><br/>
#### Before Pass Data (*BDB 2026*)

In [ ]:
# ====================================
# Load Before Pass Data
# ====================================

data_dir = Path(str(BDB_2026_BEFORE_PASS_DIR))

# ------------------------------------
# Load and Combine 18 Before Pass Datasets Into 1
# ------------------------------------

before_pass_df = (
    pl.concat(
        [
            pl.scan_csv(data_dir / f"input_2023_w{i:02d}.csv")
            .with_columns(pl.lit(i).alias("week"))
            for i in range(1, 19)
        ],
        how="vertical"
    )
    .collect()
)

# ------------------------------------
# Add Game-Play and Player-Play ID
# ------------------------------------

before_pass_df = (
    before_pass_df
    .with_columns(
        game_play_id = pl.col("game_id").cast(pl.String) + "_" + pl.col("play_id").cast(pl.String), 
        player_play_id = pl.col("game_id").cast(pl.String) + "_" + pl.col("play_id").cast(pl.String) + "_" + pl.col("nfl_id").cast(pl.String)
    )
)

# -----------------------------------
# Only Keep Players of Interest
# -----------------------------------

before_pass_df = (
    before_pass_df
    .filter(pl.col("nfl_id").is_in(players_of_interest))
)

# -----------------------------------
# Join Supplemental Data
# -----------------------------------

bdb26_df = (
    before_pass_df
    .join(
        supp_data,
        on = ["game_id", "play_id"], 
        how = "inner"
    )
)

# -----------------------------------
# Join Player-Play Data
# -----------------------------------

bdb26_df = (
    bdb26_df
    .join(
        player_play_df,
        on = ["game_id", "play_id", "nfl_id"], 
        how = "left"
    )
    # -------------------------------
    # If Player Isn't In player_play_df, We'll 
    # Assume They Weren't The Target
    # -------------------------------
    .with_columns(
        target = pl.col("target").fill_null(False)
    )
)

# -----------------------------------
# Only Keep Player-Plays Where WR Was Targeted
# -----------------------------------

# bdb26_df = (
#     bdb26_df
#     .filter(
#         pl.col("target") == True
#     )
# )

# -----------------------------------
# Only Keep Plays With Man Coverage
# -----------------------------------

bdb26_df = (
    bdb26_df
    .filter(pl.col("team_coverage_man_zone") == "MAN_COVERAGE")
)

# ----------------------------------
# Only Keep Plays That Last > 1.2 Second
# ----------------------------------

play_length_df = (
    bdb26_df
    .group_by("game_play_id")
    .agg(
        play_length_secs = pl.max("frame_id") / 10
    )
)

bdb26_df = (
    bdb26_df
    .join(
        play_length_df, 
        on = "game_play_id", 
        how = "inner"
    )
)

bdb26_df = (
    bdb26_df
    .filter(
        pl.col("play_length_secs") >= 1.2
    )
)


print("2026 BDB Data:")
bdb26_df

<br><br/>
## Examine Exposure
***

#### Combine Exposure

In [ ]:
# ====================================
# Find Percentage of Players With 
# Combine Exposure For Each Drill
# ====================================

drill_tracking_summary_df = (
    combine_tracking_df
    .filter(pl.col("nfl_id").is_in(players_of_interest))
    .group_by("drill_name")
    # -----------------------------------
    # Count Number of Players With At 
    # Least 1 Attempt In This Drill
    # -----------------------------------
    .agg(
        num_w_atl_1_attempt = pl.col("nfl_id").n_unique()
    )
    # -----------------------------------
    # Count Percentage of Players With At 
    # Least 1 Attempt In This Drill
    # -----------------------------------
    .with_columns(
        perc_w_atl_1_attempt = (
            pl.col("num_w_atl_1_attempt")
            / combine_tracking_df.select(pl.col("nfl_id").n_unique()).item()
            * 100
        )
        .round(0)
    )
    .rename({"drill_name": "drill"})
    .sort("num_w_atl_1_attempt", descending = True)

)

print("Missingness in Position-Specific Drills:")
print(f"  - Drills With At Least 80% Participation (N = {drill_tracking_summary_df.filter(pl.col("perc_w_atl_1_attempt") >= 80).height}): {drill_tracking_summary_df.filter(pl.col("perc_w_atl_1_attempt") >= 80).select("drill").to_series().to_list()}")
drill_tracking_summary_df.head(10)

<br><br/>
#### In-Game Exposure

In [ ]:
# =======================================
# Count How Many Player-Plays Worth of 
# Tracking Data Each Player Had
# =======================================

# ---------------------------------------
# Count Number of Player-Plays Played
# Among Players of Interest
# ---------------------------------------

player_play_tracking_counts_df = (
    bdb26_df
    .group_by("nfl_id")
    .agg(
        num_player_plays = pl.col("game_play_id").n_unique()
    )
)

# --------------------------------------
# Add Players With No Tracking Data
# --------------------------------------

player_play_tracking_counts_df = (
    players_df.select("nfl_id", "display_name")
    .join(
        player_play_tracking_counts_df, 
        on = "nfl_id", 
        how = "left"
    )
    .with_columns(
        num_player_plays = pl.col("num_player_plays").fill_null(0)
    )
    .sort("num_player_plays", descending = True)
)

print(f"In-Game Exposure Data:")
player_play_tracking_counts_df

In [ ]:
# =====================================
# Examine Distribution of Player-Plays
# =====================================

num_with_0_player_plays = player_play_tracking_counts_df.filter(pl.col("num_player_plays") == 0).height
perc_with_0_player_plays = round(100 * num_with_0_player_plays / player_play_tracking_counts_df.height)

num_with_atl_10_player_plays = player_play_tracking_counts_df.filter(pl.col("num_player_plays") >= 9).height
perc_with_atl_10_player_plays = round(100 * num_with_atl_10_player_plays / player_play_tracking_counts_df.height)

print(f"    - Number of Players With    0 Player-Plays: {num_with_0_player_plays} ({perc_with_0_player_plays}%)")
print(f"    - Number of Players With >= 9 Player-Plays: {num_with_atl_10_player_plays} ({perc_with_atl_10_player_plays}%)")

fig = px.box(
    player_play_tracking_counts_df, 
    y = "num_player_plays", 
    points = "all", 
    title = "Number of Player-Plays From Players of Interest"
)

fig.show()